# 04_9 – Canonical warp + nut detektálás + fret-jelöltek (HIBA 1B)

Izolált teszt: perspektíva-warp → 600×80 px canonical tér, nut detektálás, fret-vonal detektálás, és ujj-pár szűrés (`suppress_finger_pairs`).

**Cél:** Látni, hogy a canonical képen helyesen találja-e meg a nutot és a bundvonalakat. Az ujj-pár szűrés (8–22 px párok = ujj-élek) a HIBA 1B javítása.

**Ha itt hiba van:** Hamis fret-jelöltek (ujj-élek) → a spacing-fit rossz bundpozíciókat ad.

## 1. Imports, útvonalak, konstansok

A `v13`/`03c` importokat és konstansokat használjuk, változtatás nélkül.

In [ ]:
from __future__ import annotations

import math
import os
import warnings
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

try:
    import cv2
    CV2_AVAILABLE = True
    print(f"cv2 elérhető: {cv2.__version__}")
except Exception:
    cv2 = None
    CV2_AVAILABLE = False
    print("⚠️  cv2 NEM elérhető – a legtöbb lépés kihagyódik.")

try:
    import mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision as mp_vision
    MEDIAPIPE_AVAILABLE = True
    print("mediapipe elérhető")
except Exception:
    mp = mp_python = mp_vision = None
    MEDIAPIPE_AVAILABLE = False
    print("⚠️  mediapipe NEM elérhető – kézdetektálás kihagyható.")

# ── Import dari src modules ──────────────────────────────────────────────────
from src.hand_landmark import build_finger_mask, anchor_neck_angle, FINGER_CHAINS, FINGER_THICK_MULT
from src.geometry import step1_canny, step2_hough, step3_neck_angle, step3_neck_angle_anchored, step4_split_lines, step5_outer_edges

# ── Projekt útvonalak ─────────────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT     = PROJECT_ROOT / "data"
NOTEBOOK_DIR  = PROJECT_ROOT / "notebooks"
MODEL_DIR     = PROJECT_ROOT / "models"
MANIFEST_PATH = DATA_ROOT / "split_manifest.csv"
OUTPUT_DIR    = PROJECT_ROOT / "output" / "03b_pipeline_debug"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Geometriai konstansok ──────────────────────────────────────────────────
CANONICAL_W   = 600          # kanonikus tér szélessége (px)
CANONICAL_H   = 80           # kanonikus tér magassága (px)
N_FRETS       = 24
FRET_RULE     = 17.817

# Bund n x-pozíciója ha a teljes nyak kitölti a kanonikus teret [0..N_FRETS]
# x_n = CANONICAL_W * (1 - 0.5^(n/12)) / 0.75
FRET_POS_FULL = np.array(
    [CANONICAL_W * (1.0 - 0.5 ** (n / 12.0)) / 0.75 for n in range(N_FRETS + 1)],
    dtype=np.float64,
)
# Normalizált pozíciók (0..1, ahol 1.0 = a 24. bund pozíciója)
FRET_POS_NORM = np.array(
    [1.0 - 0.5 ** (n / 12.0) for n in range(N_FRETS + 1)],
    dtype=np.float64,
)

FINGER_TIP_IDX = [4, 8, 12, 16, 20]   # MediaPipe ujjhegy landmark indexek
INLAY_FRETS    = [3, 5, 7, 9, 12]     # Standard 5 inlay pozíció
INLAY_FRETS_FULL = [3, 5, 7, 9, 12, 15, 17, 19, 21, 24]  # Mindkét oktáv

# Inlay normalizált x-pozíciók: az n. inlay a FRET_POS_NORM[n-1] és [n] közötti tér közepe
INLAY_NORM_DICT = {
    n: (FRET_POS_NORM[n - 1] + FRET_POS_NORM[n]) / 2.0
    for n in INLAY_FRETS_FULL
}

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"Project root : {PROJECT_ROOT}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"FRET_POS_FULL[:7] = {np.round(FRET_POS_FULL[:7], 1)}")

## 2. Seed-ek beállítása — HIBA 2 keret-rendszer

`FIXED_SEEDS = [42, 43]` rögzített, a másik **8 seed** futásonként random
(`np.random.default_rng()` belső állapotából). A teljes lista kiírásra kerül,
hogy ha valami furcsa eredmény jön, azt egy fix seedlistával reprodukálni lehessen.


In [ ]:
# HIBA 2: 10 kép – 2 fix + 8 random seed
FIXED_SEEDS = [42, 43]
_seed_rng   = np.random.default_rng()   # nincs seed → futásonként más
RANDOM_SEEDS = [int(_seed_rng.integers(0, 1_000_000)) for _ in range(8)]
ALL_SEEDS    = FIXED_SEEDS + RANDOM_SEEDS
print("ALL_SEEDS (notebook tetején):", ALL_SEEDS)


## 3. 10 kép mintavételezése a manifest-ből

A `data/split_manifest.csv` "train" split-jéből minden seed-hez 1 random képet
veszünk (`pandas.DataFrame.sample(n=1, random_state=seed)`). Így a fix seed-ek
ugyanazt a 2 képet adják minden futáskor, a 8 random pedig más-más képet.


In [ ]:
# Manifest betöltés + helper függvények (v13 cell 4 alapján, módosítva)
manifest = pd.read_csv(MANIFEST_PATH)
CLASSES  = sorted(manifest["class"].unique())
print(f"Összes kép: {len(manifest)}, osztályok: {CLASSES}")

def load_image_bgr(img_path: str):
    """BGR kép betöltése (cv2 elsődleges, PIL fallback)."""
    if CV2_AVAILABLE:
        img = cv2.imread(str(img_path))
        if img is not None:
            return img
    if PIL_AVAILABLE:
        arr = np.array(Image.open(img_path).convert("RGB"))
        return arr[:, :, ::-1].copy()
    return None

def bgr2rgb(img_bgr):
    if CV2_AVAILABLE:
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return img_bgr[:, :, ::-1]

# Seedenként 1 random train-kép
train_pool = manifest[manifest["split"] == "train"].reset_index(drop=True)
IMAGES = []
for s in ALL_SEEDS:
    row = train_pool.sample(n=1, random_state=s).iloc[0]
    IMAGES.append({
        "seed":  s,
        "path":  str(row["path"]),
        "class": row["class"],
        "fname": Path(row["path"]).name,
    })

for img in IMAGES:
    print(f"  seed={img['seed']:>6}  class={img['class']}  file={img['fname']}")


## 2. MediaPipe model inicializálás

A `hand_landmarker.task` modell betöltése. Ha nem létezik, automatikusan letölti.

In [ ]:
# MediaPipe model inicializálás
HAND_LANDMARKER = None
if MEDIAPIPE_AVAILABLE:
    model_path = MODEL_DIR / "hand_landmarker.task"
    if not model_path.exists():
        import urllib.request
        model_path.parent.mkdir(parents=True, exist_ok=True)
        MODEL_URL = (
            "https://storage.googleapis.com/mediapipe-models/"
            "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
        )
        print(f"MediaPipe model letöltése: {MODEL_URL}")
        urllib.request.urlretrieve(MODEL_URL, model_path)
    try:
        opts = mp_vision.HandLandmarkerOptions(
            base_options=mp_python.BaseOptions(model_asset_path=str(model_path)),
            running_mode=mp_vision.RunningMode.IMAGE,
            num_hands=2,
        )
        HAND_LANDMARKER = mp_vision.HandLandmarker.create_from_options(opts)
        print("✅ HandLandmarker inicializálva.")
    except Exception as exc:
        print(f"⚠️  HandLandmarker hiba: {exc}")


def step9_detect_landmarks(img_path: str, landmarker=None):
    """MediaPipe kéz landmark detektálás. Visszaad: [(x_norm,y_norm,z_norm)]*21 vagy None."""
    if landmarker is None or not MEDIAPIPE_AVAILABLE:
        return None
    try:
        image  = mp.Image.create_from_file(str(img_path))
        result = landmarker.detect(image)
        if not result.hand_landmarks:
            return None
        best = result.hand_landmarks[0]
        return [(float(lm.x), float(lm.y), float(lm.z)) for lm in best]
    except Exception as exc:
        print(f"  [mediapipe] hiba: {exc}")
        return None


def step9_project_fingertips(landmarks, H: np.ndarray,
                             w_img: int, h_img: int,
                             fit: dict = None) -> list[dict]:
    """
    21 MediaPipe landmark → kanonikus tér vetítés.

    Visszaad: lista dict-ekből, minden ujjhegyhez:
        tip_idx       : int – MediaPipe index (4,8,12,16,20)
        canon_x, canon_y : float – kanonikus px
        string_norm   : float 0–1 – húr pozíció
        fret_est      : int|None – legközelebbi illesztett bund
    """
    if landmarks is None or H is None:
        return []
    results = []
    pred = fit.get("predicted_x", {}) if fit else {}
    for tip_idx in FINGER_TIP_IDX:
        if tip_idx >= len(landmarks):
            continue
        xn, yn, _ = landmarks[tip_idx]
        px, py    = xn * w_img, yn * h_img
        pt        = np.array([px, py, 1.0])
        proj      = H @ pt
        if abs(proj[2]) < 1e-9:
            continue
        cx = float(proj[0] / proj[2])
        cy = float(proj[1] / proj[2])
        str_norm  = float(np.clip(cy / CANONICAL_H, 0.0, 1.0))
        fret_est  = None
        if pred:
            fret_est = min(pred.keys(), key=lambda n: abs(pred[n] - cx))
        results.append({
            "tip_idx":   tip_idx,
            "canon_x":   cx,
            "canon_y":   cy,
            "string_norm": str_norm,
            "fret_est":  fret_est,
        })
    return results


def viz_fingertips_canonical(canon_bgr: np.ndarray, tips: list[dict],
                              fit: dict = None) -> None:
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.imshow(bgr2rgb(canon_bgr), aspect="auto")
    if fit:
        for n, x in fit.get("predicted_x", {}).items():
            if 0 <= x <= CANONICAL_W:
                ax.axvline(x, color="lime", lw=0.6, alpha=0.4, linestyle=":")
                ax.text(x, 3, str(n), color="lime", fontsize=6,
                        ha="center", va="top")
    colors = ["#FF6B6B","#FFD93D","#6BCB77","#4D96FF","#F4A261"]
    finger_names = {4:"Hüvelyk",8:"Mutató",12:"Középső",16:"Gyűrűs",20:"Kisujj"}
    for tip, col in zip(tips, colors):
        if -5 <= tip["canon_x"] <= CANONICAL_W+5 and 0 <= tip["canon_y"] <= CANONICAL_H:
            ax.plot(tip["canon_x"], tip["canon_y"], "o", color=col, ms=10,
                    zorder=6, label=f"{finger_names.get(tip['tip_idx'],tip['tip_idx'])} → bund {tip['fret_est']}")
    ax.legend(loc="lower right", fontsize=8, ncol=2)
    ax.set_title("STEP 9: Ujjhegyek a kanonikus térben")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def viz_overlay_original(img_bgr: np.ndarray, corners: np.ndarray,
                         H_inv: np.ndarray, tips: list[dict],
                         fit: dict, landmarks) -> None:
    """Teljes overlay az eredeti képen: trapéz + bund-vonalak visszavetítve + ujjhegyek."""
    if not CV2_AVAILABLE:
        return
    overlay = bgr2rgb(img_bgr).copy()
    h_img, w_img = img_bgr.shape[:2]
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.imshow(overlay)
    # Trapezoid
    loop = np.vstack([corners, corners[0]])
    ax.plot(loop[:,0], loop[:,1], color="orange", lw=2.5, label="Trapezoid")
    # Bund vonalak visszavetítve a képtérbe
    for n, x_c in fit.get("predicted_x", {}).items():
        if not (0 <= x_c <= CANONICAL_W):
            continue
        top_c    = np.array([x_c, 0.0, 1.0])
        bot_c    = np.array([x_c, float(CANONICAL_H), 1.0])
        top_orig = H_inv @ top_c
        bot_orig = H_inv @ bot_c
        if abs(top_orig[2]) < 1e-9 or abs(bot_orig[2]) < 1e-9:
            continue
        tx, ty = top_orig[0]/top_orig[2], top_orig[1]/top_orig[2]
        bx, by = bot_orig[0]/bot_orig[2], bot_orig[1]/bot_orig[2]
        col = "yellow" if n in INLAY_FRETS else "cyan"
        ax.plot([tx,bx],[ty,by], color=col, lw=0.8, alpha=0.7)
        ax.text(tx, ty-5, str(n), color=col, fontsize=7, ha="center")
    # MediaPipe ujjhegyek (ha van)
    if landmarks:
        colors = ["#FF6B6B","#FFD93D","#6BCB77","#4D96FF","#F4A261"]
        for tip, col in zip(tips, colors):
            xi = tip["canon_x"]
            yi = tip["canon_y"]
            pt_c = np.array([xi, yi, 1.0])
            pt_o = H_inv @ pt_c
            if abs(pt_o[2]) < 1e-9:
                continue
            px, py = pt_o[0]/pt_o[2], pt_o[1]/pt_o[2]
            ax.plot(px, py, "o", color=col, ms=12, zorder=7,
                    label=f"ujj→bund {tip['fret_est']}")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title("STEP 9 overlay: Visszavetített bundok + ujjhegyek")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


## 2. Függvények

## 5. Új helper függvények — HIBA 1 és HIBA 3 javítás

### 5.1 `build_finger_mask` — HIBA 1A

MediaPipe 21 landmarkjából a 4 ujj (mutató, középső, gyűrűs, kisujj) és a
hüvelykujj szegmenseihez vastag vonalakat rajzol egy `uint8` maszkra. A
szegmens-vastagság a szegmens hosszának 18–22%-a (gitár-ujj arány), a hüvelyk
kicsit vastagabb. A maszkot a Canny edge map-re `& ~mask` formában alkalmazzuk
a `run_v14_pipeline`-ban, így a Hough nem lát ujj-éleket.


### 5.2 `anchor_neck_angle` — HIBA 3A

A landmark-okból (csukló = 0, középső-MCP = 9, mutató-MCP = 5, gyűrűs-MCP = 13)
becsüljük a kéz fő tengelyét és a tenyér centroidját. A gitárnyak iránya közel
**merőleges** erre a kéztengelyre, így ez egy erős prior a step3 hisztogramra:
`expected_neck_angle ± 25°` ablakon belüli legerősebb csúcsot vesszük.


In [ ]:
def _pt_on_line(midpoint: np.ndarray, neck_dir: np.ndarray,
                target_along: float) -> np.ndarray:
    """Egy vonalon lévő pont, amelynek along-vetülete = target_along."""
    base = float(np.dot(midpoint, neck_dir))
    return midpoint + (target_along - base) * neck_dir


def step6_trapezoid(img_bgr: np.ndarray, edge_info: dict) -> Optional[dict]:
    """
    A két kiválasztott él alapján felépíti a fretboard trapézot (v9).

    v9 változás (Fix C):
      - a_min/a_max immár csak az `inner_lines` (left_cl + right_cl) végpontjaiból
        számolódik, NEM az `all_projections`-ből. A strap/póló-élek nem nyújtják
        ki a trapézt a fogólap fizikai hosszán túl.

    v8 változás megmaradt:
      - Orientáció-döntés 5%-os hiszterezissel

    Visszaad:
        corners_px  : np.float32 (4,2) – [TL, TR, BR, BL]
        area_px2    : float
        w_start, w_end : float – nyakszélesség a két végponton
    """
    if edge_info is None:
        return None
    edges    = edge_info["selected_edges"]
    neck_dir = edge_info["neck_dir"]
    perp_dir = edge_info["perp_dir"]
    if len(edges) < 2:
        return None

    left, right = sorted(edges, key=lambda e: e["proj"])

    # v9 Fix C: a_min/a_max csak az inner_lines végpontjaiból
    # (a klaszterhez tartozó húr-szegmensek; nincs strap/póló-szennyezés)
    inner = edge_info.get("inner_lines") or edge_info.get("all_projections")
    all_endpts = []
    for item in inner:
        x1, y1, x2, y2 = item["line"]
        all_endpts.append(np.array([x1, y1], dtype=np.float64))
        all_endpts.append(np.array([x2, y2], dtype=np.float64))
    alongs = [float(np.dot(p, neck_dir)) for p in all_endpts]
    a_min  = min(alongs)
    a_max  = max(alongs)

    # Kis margó mindkét végén (2%)
    span  = a_max - a_min
    a_min -= span * 0.02
    a_max += span * 0.02

    l_start = _pt_on_line(left["midpoint"],  neck_dir, a_min)
    l_end   = _pt_on_line(left["midpoint"],  neck_dir, a_max)
    r_start = _pt_on_line(right["midpoint"], neck_dir, a_min)
    r_end   = _pt_on_line(right["midpoint"], neck_dir, a_max)

    w_start = float(np.linalg.norm(r_start - l_start))
    w_end   = float(np.linalg.norm(r_end   - l_end))

    # Orientáció-döntés 5%-os hiszterezissel (v8 Fix 4 marad)
    hysteresis = 0.05 * min(w_start, w_end)
    if (w_end - w_start) >= -hysteresis:   # w_start szűkebb vagy közel egyenlő
        tl, tr, br, bl = l_start, l_end, r_end, r_start
    else:
        tl, tr, br, bl = l_end, l_start, r_start, r_end

    corners = np.array([tl, tr, br, bl], dtype=np.float32)
    area = 0.5 * abs(
        np.dot(corners[:,0], np.roll(corners[:,1],-1)) -
        np.dot(corners[:,1], np.roll(corners[:,0],-1))
    )
    print(f"  [trapezoid_v9] span={span:.1f}px | "
          f"w_start={w_start:.1f}px | w_end={w_end:.1f}px | area={area:.0f}px² | "
          f"forrás: inner_lines ({len(inner)} szegmens)")
    return {"corners_px": corners, "area_px2": float(area),
            "w_start": w_start, "w_end": w_end}


def step6_warp(img_bgr: np.ndarray, corners_px: np.ndarray
               ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Perspektíva-warp a trapézból a kanonikus 600×80 px térbe.

    Visszaad: (H, H_inv, canon_bgr)
    """
    if not CV2_AVAILABLE:
        raise RuntimeError("cv2 szükséges")
    dst = np.array([
        [0,             0            ],  # source TL → canonical top-left
        [CANONICAL_W-1, 0            ],  # source TR → canonical top-right
        [CANONICAL_W-1, CANONICAL_H-1],  # source BR → canonical bottom-right
        [0,             CANONICAL_H-1],  # source BL → canonical bottom-left
    ], dtype=np.float32)
    H     = cv2.getPerspectiveTransform(corners_px.astype(np.float32), dst)
    H_inv = cv2.getPerspectiveTransform(dst, corners_px.astype(np.float32))
    canon = cv2.warpPerspective(img_bgr, H, (CANONICAL_W, CANONICAL_H))
    return H, H_inv, canon


def viz_trapezoid(img_bgr: np.ndarray, corners: np.ndarray,
                  canon_bgr: np.ndarray) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.imshow(bgr2rgb(img_bgr))
    loop = np.vstack([corners, corners[0]])
    ax.plot(loop[:,0], loop[:,1], color="orange", lw=2.5)
    for i, (pt, lbl) in enumerate(zip(corners, ["TL","TR","BR","BL"])):
        ax.plot(pt[0], pt[1], "o", ms=8, color="orange", zorder=5)
        ax.annotate(lbl, pt, textcoords="offset points", xytext=(5,5),
                    fontsize=8, color="orange", fontweight="bold")
    ax.set_title("STEP 6a [v9]: Trapezoid a képtérben")
    ax.axis("off")
    ax = axes[1]
    ax.imshow(bgr2rgb(canon_bgr), aspect="auto")
    ax.set_title(f"STEP 6b [v9]: Kanonikus tér ({CANONICAL_W}×{CANONICAL_H} px)")
    ax.set_xlabel("Kanonikus x (px) → nyak hossza")
    ax.set_ylabel("Kanonikus y (px) → húrok")
    plt.tight_layout()
    plt.show()


In [ ]:
def validate_trapezoid(corners, img_shape, landmarks=None,
                       min_aspect: float = 4.0,
                       area_frac_range: tuple = (0.015, 0.45),
                       hand_inside_frac: float = 0.15,
                       max_edge_angle_diff_deg: float = 15.0):
    """Trapéz épelméjűségi szanitás.

    A 4 oldal-hossz közül a 2 hosszabb pár átlaga adja a long_axis-t, a 2
    rövidebb pár átlaga a short_axis-t — így a tényleges corner-ordering
    (TL/TR/BR/BL melyik tengelyhez melyik él tartozik) MINDEGY.

    Returns:
        (ok: bool, reasons: list[str]) – ok=False esetén a reasons sorolja
        a bukott szabályokat.
    """
    h, w = img_shape[:2]
    img_area = float(h * w)
    reasons = []

    corners = np.asarray(corners, dtype=np.float64).reshape(4, 2)
    tl, tr, br, bl = corners

    # Oldalak hossza (mind a 4) → long_axis = leghosszabbak átlaga
    edge_lens = sorted([
        float(np.linalg.norm(tr - tl)),
        float(np.linalg.norm(br - tr)),
        float(np.linalg.norm(bl - br)),
        float(np.linalg.norm(tl - bl)),
    ])
    short_axis = (edge_lens[0] + edge_lens[1]) / 2.0
    long_axis  = (edge_lens[2] + edge_lens[3]) / 2.0
    aspect = long_axis / max(short_axis, 1e-3)
    if aspect < min_aspect:
        reasons.append(f"aspect {aspect:.2f}<{min_aspect}")

    # Terület (shoelace)
    area = 0.5 * abs(
        np.dot(corners[:, 0], np.roll(corners[:, 1], -1)) -
        np.dot(corners[:, 1], np.roll(corners[:, 0], -1))
    )
    frac = area / img_area
    if not (area_frac_range[0] <= frac <= area_frac_range[1]):
        reasons.append(f"area_frac {frac:.3f} ∉ {area_frac_range}")

    # Hosszú élek párhuzamossága: a két leghosszabb él egymással ≤ X°
    # Erre kell egy él-indexet visszafejteni → vektoros megközelítés
    edge_vecs = [tr - tl, br - tr, bl - br, tl - bl]
    edge_pairs = sorted(enumerate(edge_vecs),
                        key=lambda iv: -float(np.linalg.norm(iv[1])))[:2]
    v_a, v_b = edge_pairs[0][1], edge_pairs[1][1]
    ang_a = np.degrees(np.arctan2(v_a[1], v_a[0]))
    ang_b = np.degrees(np.arctan2(v_b[1], v_b[0]))
    # Cirkuláris távolság 180°-os szimmetriával (irány-jel mindegy)
    diff = abs(((ang_a - ang_b + 90.0) % 180.0) - 90.0)
    if diff > max_edge_angle_diff_deg:
        reasons.append(f"edges_angle_diff {diff:.1f}°>"
                       f"{max_edge_angle_diff_deg}°")

    # Lefedi-e a kezet (ha van landmark)
    if landmarks is not None and CV2_AVAILABLE:
        pts_px = np.array([[lx * w, ly * h] for (lx, ly, _) in landmarks],
                          dtype=np.float32)
        contour = corners.astype(np.float32)
        inside = sum(
            1 for p in pts_px
            if cv2.pointPolygonTest(contour, (float(p[0]), float(p[1])), False) >= 0
        )
        frac_in = inside / float(len(pts_px))
        if frac_in < hand_inside_frac:
            reasons.append(f"hand_inside {frac_in:.2f}<{hand_inside_frac}")

    return (len(reasons) == 0), reasons


In [ ]:
def step6b_find_nut(canon_bgr: np.ndarray,
                    search_frac: float = 0.30,
                    threshold_factor: float = 2.5,
                    min_offset: int = 5
                    ) -> Optional[dict]:
    """
    Megkeresi a nutot (0. bund) a kanonikus képen.

    A nut a fogólap kezdetét jelző világos csont/műanyag/sárgaréz vonal –
    a sötét fa fogólapon nagy gradiens-választ ad. Sobel-x oszlopösszege
    erős csúcsot ad ott, ahol a nut van.

    Stratégia:
      1. Vízszintes gradiens (|Sobel-x|) oszlopösszege a teljes canon-on
      2. Keresés a bal és jobb széli `search_frac` sávban
      3. A két oldal közül a stronger peak nyer (a fit_direction-tól
         függetlenül – így forward és reversed esetre is működik)
      4. Acceptance: peak ≥ medián × threshold_factor

    Visszaad:
        None ha nincs egyértelmű nut, vagy
        {"side": "left"|"right", "nut_x": int, "peak": float,
         "ratio": float, "col_response": np.ndarray}
    """
    if not CV2_AVAILABLE:
        return None
    gray = cv2.cvtColor(canon_bgr, cv2.COLOR_BGR2GRAY)
    sx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    col_response = np.abs(sx).sum(axis=0).astype(np.float32)

    w = canon_bgr.shape[1]
    sw = max(int(w * search_frac), 10)
    # Csípjük le a legszélső pixeleket (Sobel-él-artifaktok)
    left_region  = col_response[min_offset : sw]
    right_region = col_response[w - sw : w - min_offset]

    if len(left_region) == 0 or len(right_region) == 0:
        return None

    left_peak_val   = float(left_region.max())
    right_peak_val  = float(right_region.max())
    median_response = float(np.median(col_response))

    threshold = threshold_factor * (median_response + 1e-6)

    if left_peak_val >= right_peak_val:
        side    = "left"
        peak    = left_peak_val
        nut_x   = int(np.argmax(left_region)) + min_offset
    else:
        side    = "right"
        peak    = right_peak_val
        nut_x   = (w - sw) + int(np.argmax(right_region))

    ratio = peak / (median_response + 1e-6)

    print(f"  [nut_detect_v10] median={median_response:.0f} | "
          f"left_peak={left_peak_val:.0f}@{int(np.argmax(left_region))+min_offset}px | "
          f"right_peak={right_peak_val:.0f}@{(w-sw)+int(np.argmax(right_region))}px")

    if peak < threshold:
        print(f"  [nut_detect_v10] nincs egyértelmű nut (peak/median={ratio:.2f} < {threshold_factor})")
        return None

    print(f"  [nut_detect_v10] nut találat: {side} oldal @ x={nut_x}px "
          f"(peak/median={ratio:.2f})")

    return {"side": side, "nut_x": nut_x, "peak": peak, "ratio": ratio,
            "col_response": col_response}


def step6c_trim_to_nut(corners_px: np.ndarray,
                       H_inv: np.ndarray,
                       nut_info: dict) -> np.ndarray:
    """
    A trapezoid sarkainak áthelyezése a nut canonical x-pozíciójához
    tartozó képtér-pontokra.

    - side="left" → TL és BL átkerül (nut_x, 0) és (nut_x, CANONICAL_H-1) képbeli pontjára
    - side="right" → TR és BR átkerül ugyanígy

    Visszaad: új corners_px (4×2 np.float32)
    """
    nut_x = nut_info["nut_x"]
    side  = nut_info["side"]

    canon_pts = np.array([[nut_x, 0],
                           [nut_x, CANONICAL_H - 1]], dtype=np.float32)
    canon_pts = canon_pts.reshape(-1, 1, 2)
    img_pts = cv2.perspectiveTransform(canon_pts, H_inv).reshape(-1, 2)

    new_corners = corners_px.copy().astype(np.float32)
    if side == "left":
        new_corners[0] = img_pts[0]   # TL → (nut_x, 0) képbe
        new_corners[3] = img_pts[1]   # BL → (nut_x, CANONICAL_H-1) képbe
    else:  # right
        new_corners[1] = img_pts[0]   # TR
        new_corners[2] = img_pts[1]   # BR
    return new_corners


def viz_nut_detection(canon_bgr: np.ndarray, nut_info: Optional[dict]) -> None:
    """A kanonikus kép + oszlopgradiens-görbe + detektált nut x."""
    fig, axes = plt.subplots(2, 1, figsize=(13, 5),
                              gridspec_kw={"height_ratios": [2, 1]})
    ax = axes[0]
    ax.imshow(bgr2rgb(canon_bgr), aspect="auto")
    if nut_info:
        nx = nut_info["nut_x"]
        col = "magenta" if nut_info["side"] == "left" else "cyan"
        ax.axvline(nx, color=col, lw=2.0, alpha=0.9,
                   label=f'nut @ x={nx} ({nut_info["side"]}, ratio={nut_info["ratio"]:.1f})')
        ax.legend(loc="upper right", fontsize=9)
    ax.set_title("STEP 6b [v10]: Nut detektálás a kanonikus képen")
    ax.axis("off")

    ax = axes[1]
    if nut_info:
        cr = nut_info["col_response"]
        ax.plot(cr, color="black", lw=1.2)
        ax.axhline(np.median(cr), color="gray", ls="--", lw=0.8,
                   label=f"median={np.median(cr):.0f}")
        ax.axvline(nut_info["nut_x"], color="magenta" if nut_info["side"]=="left" else "cyan",
                   lw=2.0, alpha=0.85)
        ax.set_xlim(0, len(cr))
        ax.set_ylabel("Σ|∂x|")
        ax.set_xlabel("Canonical x (px)")
        ax.legend(loc="upper right", fontsize=8)
    else:
        ax.text(0.5, 0.5, "Nincs nut detektálva", ha="center", va="center",
                transform=ax.transAxes, fontsize=11, color="gray")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
def _column_variance_frets(canon_bgr: np.ndarray,
                            min_height: float = 0.18,
                            min_distance: int = 10,
                            smooth_kernel: int = 5) -> list[float]:
    """
    Fallback bunddetektálás oszlopvariancia alapján.

    A kanonikus képen oszloponként kiszámítja a pixelszórást.
    A bund-vonalak mentén az intenzitás hirtelen változik → lokális szórás-csúcsok.

    Visszaad: detektált x-pozíciók listája (px).
    """
    if not CV2_AVAILABLE:
        return []
    gray     = cv2.cvtColor(canon_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    col_std  = np.std(gray, axis=0)           # shape: (CANONICAL_W,)
    max_val  = col_std.max()
    if max_val < 1e-3:
        return []
    col_norm = col_std / max_val              # 0..1

    # Simítás
    k = smooth_kernel if smooth_kernel % 2 else smooth_kernel + 1
    kernel   = np.ones(k) / k
    smoothed = np.convolve(col_norm, kernel, mode="same")

    # Csúcskereső (scipy ha elérhető, különben saját)
    peaks = []
    try:
        from scipy.signal import find_peaks as _fp
        idxs, _ = _fp(smoothed, height=min_height, distance=min_distance)
        peaks = [float(i) for i in idxs]
    except ImportError:
        # Egyszerű lokális maximum keresés
        for i in range(1, len(smoothed) - 1):
            if smoothed[i] > smoothed[i-1] and smoothed[i] > smoothed[i+1] \
                    and smoothed[i] > min_height:
                peaks.append(float(i))
        # min_distance szűrés
        filtered = []
        for p in peaks:
            if not filtered or p - filtered[-1] >= min_distance:
                filtered.append(p)
        peaks = filtered

    print(f"  [col_var_fallback] csúcsok: {len(peaks)}")
    return peaks


def _filter_wide_clusters(clusters: list,
                           max_width_frac: float,
                           max_fret_width_px: float) -> tuple[list, list]:
    """
    Szélességalapú klaszterszűrő (v12).

    Egy klaszter x-kiterjedése nem haladhatja meg:
      - az összes klaszterközép medián-távolságának `max_width_frac`-szorosát, VAGY
      - az abszolút `max_fret_width_px` korlátot.
    Amelyik kisebb, az dönt.

    Ha az összes klaszter kizárásra kerülne → visszaadja az eredetit (safe fallback).

    Visszaad: (filtered_clusters, rejected_widths)
    """
    if len(clusters) < 2:
        return clusters, []

    means = sorted([float(np.mean(c)) for c in clusters])
    spacings = np.diff(means)
    median_spacing = float(np.median(spacings)) if len(spacings) > 0 else float("inf")

    adaptive_limit = max_width_frac * median_spacing if median_spacing > 1e-3 else float("inf")
    width_limit    = min(adaptive_limit, max_fret_width_px)

    kept, rejected_widths = [], []
    for c in clusters:
        w = float(max(c) - min(c)) if len(c) > 1 else 0.0
        if w <= width_limit:
            kept.append(c)
        else:
            rejected_widths.append(w)
            print(f"  [step7_v12] Klaszter kizárva (szélesség {w:.1f}px "
                  f"> limit {width_limit:.1f}px, medián rés: {median_spacing:.1f}px)")

    if not kept:
        print(f"  [step7_v12] ⚠ Összes klaszter kizárásra kerülne → szűrő kikapcsolva (fallback)")
        return clusters, []

    return kept, rejected_widths


def step7_fret_lines_canonical(canon_bgr: np.ndarray,
                               canny_low: int = 15,
                               canny_high: int = 75,
                               threshold: int = 18,
                               min_len_frac: float = 0.25,
                               max_gap: int = 8,
                               angle_tol_from_vert: float = 45.0,
                               cluster_gap: float = 15.0,
                               var_fallback: bool = True,
                               var_min_height: float = 0.18,
                               var_min_distance: int = 10,
                               max_width_frac: float = 0.4,
                               max_fret_width_px: float = 18.0) -> list[float]:
    """
    Bundvonalak detektálása kanonikus 600×80 px képen (v5).

    v5 változás: szélességalapú klasztszűrő hozzáadva.
      Paraméterek:
        max_width_frac    – klaszter-szélesség / medián-bundköz max aránya (default 0.4)
        max_fret_width_px – abszolút max klaszterszélesség px-ben (default 18 px)

    Szint 1: HoughLinesP lazább paraméterekkel.
    Szint 2 (fallback): oszlopvariancia alapú csúcskereső.

    Visszaad: klaszterezett x-pozíciók listája (px).
    """
    if not CV2_AVAILABLE:
        raise RuntimeError("cv2 szükséges")
    gray    = cv2.cvtColor(canon_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (3,3), 0)
    edges   = cv2.Canny(blurred, canny_low, canny_high)
    lines   = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi/180, threshold=threshold,
        minLineLength=max(int(CANONICAL_H * min_len_frac), 6),
        maxLineGap=max_gap,
    )

    xs_all = []
    xs_ok  = []
    if lines is not None:
        for ln in lines:
            x1,y1,x2,y2 = map(int, ln[0])
            angle_v = abs(90.0 - abs(np.degrees(np.arctan2(y2-y1, x2-x1))))
            xs_all.append((x1+x2)/2.0)
            if angle_v < angle_tol_from_vert:
                xs_ok.append((x1+x2)/2.0)

    print(f"  [step7] Hough: {len(xs_all)} nyers vonal → szűrve: {len(xs_ok)}")

    # Ha Hough talált elegendő vonalat → klaszterezés + szélességszűrés
    if xs_ok:
        xs_ok.sort()
        clusters = [[xs_ok[0]]]
        for x in xs_ok[1:]:
            if abs(x - clusters[-1][-1]) < cluster_gap:
                clusters[-1].append(x)
            else:
                clusters.append([x])

        # v12: szélességalapú szűrő – ujjak és kézkörvonalak kizárása
        clusters, rejected = _filter_wide_clusters(clusters, max_width_frac, max_fret_width_px)

        result = [float(np.mean(c)) for c in clusters]
        print(f"  [step7] HoughLinesP → {len(result)} klaszter "
              f"({len(rejected)} széles kizárva)")
        return result

    # ── Fallback: oszlopvariancia ─────────────────────────────────────────
    if var_fallback:
        print("  [step7] Hough 0 vonalat adott → oszlopvariancia fallback")
        result = _column_variance_frets(
            canon_bgr, min_height=var_min_height,
            min_distance=var_min_distance
        )
        return result

    return []


def viz_fret_lines_canonical(canon_bgr: np.ndarray,
                              detected_x: list[float],
                              matched_frets: dict = None,
                              title_prefix: str = "STEP 7") -> None:
    fig, axes = plt.subplots(2, 1, figsize=(14, 5),
                              gridspec_kw={"height_ratios": [3, 1]})

    # Felső: kanonikus kép + detektált vonalak
    ax = axes[0]
    ax.imshow(bgr2rgb(canon_bgr), aspect="auto")
    for x in FRET_POS_FULL:
        if 0 <= x <= CANONICAL_W:
            ax.axvline(x, color="white", lw=0.6, alpha=0.3, linestyle="--")
    for x in detected_x:
        ax.axvline(x, color="tomato", lw=1.8, alpha=0.9)
    if matched_frets:
        for n, x in matched_frets.items():
            ax.axvline(x, color="lime", lw=0.8, alpha=0.5, linestyle=":")
            ax.text(x, 3, str(n), color="lime", fontsize=6,
                    ha="center", va="top", fontweight="bold")
    ax.set_title(f"{title_prefix}: kanonikus kép | detektált: {len(detected_x)}")
    ax.axis("off")

    # Alsó: oszlopvariancia görbe
    ax2 = axes[1]
    gray    = cv2.cvtColor(canon_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    col_std = np.std(gray, axis=0)
    max_v   = col_std.max() + 1e-6
    xs_plot = np.arange(len(col_std))
    ax2.fill_between(xs_plot, col_std/max_v, alpha=0.4, color="steelblue")
    ax2.plot(xs_plot, col_std/max_v, color="steelblue", lw=0.8)
    for x in detected_x:
        ax2.axvline(x, color="tomato", lw=1.5, alpha=0.8)
    ax2.set_xlim(0, CANONICAL_W)
    ax2.set_ylim(0, 1.05)
    ax2.set_xlabel("Kanonikus x (px)")
    ax2.set_ylabel("Norm. szórás")
    ax2.set_title("Oszlopvariancia görbe")
    plt.tight_layout()
    plt.show()


### 5.4 `suppress_finger_pairs` — HIBA 1B

A canonical térben (600×80 px) a frets közötti minimum spacing kb. **30+ px**
(a legkeskenyebb bundköz a 24. bund környékén). Ezzel szemben **8–22 px**
távolságra **csak** ujj-él párok lehetnek (egy ujj két szélén). Ha találunk
ilyen pár-t, **mindkét** jelöltet elvetjük (egy ujj-él NEM bund).


In [ ]:
def suppress_finger_pairs(fret_xs, min_px: float = 8.0, max_px: float = 22.0):
    """Eltávolítja azokat a fret-jelölteket, amelyek 8–22 px-en belül egy másikkal párban vannak.

    Returns:
        (kept: list[float], removed_pairs: list[tuple[float, float]])
    """
    if len(fret_xs) < 2:
        return list(fret_xs), []
    xs = np.array(sorted(fret_xs), dtype=np.float64)
    removed_idx = set()
    pairs = []
    for i in range(len(xs)):
        if i in removed_idx:
            continue
        for j in range(i + 1, len(xs)):
            if j in removed_idx:
                continue
            d = xs[j] - xs[i]
            if d > max_px:
                break
            if min_px <= d <= max_px:
                removed_idx.add(i)
                removed_idx.add(j)
                pairs.append((float(xs[i]), float(xs[j])))
                break
    kept = [float(x) for k, x in enumerate(xs) if k not in removed_idx]
    return kept, pairs


## 3. Futtatás – 10 kép

In [ ]:
# ── Futtatás: Canonical warp + nut + fret jelöltek 10 képen ──────────────
def _run_to_frets(entry):
    r = {"seed": entry["seed"], "class": entry["class"],
         "fname": entry["fname"], "ok": False}
    img = load_image_bgr(entry["path"])
    if img is None:
        r["invalid_reason"] = "load_failed"; return r
    r["img"] = img
    lms    = step9_detect_landmarks(entry["path"], HAND_LANDMARKER)
    anchor = anchor_neck_angle(lms, img.shape)
    mask   = build_finger_mask(img.shape, lms)
    edges_m = step1_canny(img)
    if mask.any():
        edges_m[mask > 0] = 0
    lines = step2_hough(img, edges_m)
    if not lines:
        r["invalid_reason"] = "no_hough_lines"; return r
    neck_p = step3_neck_angle(lines)
    neck_a = step3_neck_angle_anchored(lines, anchor=anchor) if anchor else neck_p
    sp = step4_split_lines(lines, neck_p["angle_deg"])
    sa = step4_split_lines(lines, neck_a["angle_deg"])
    neck = neck_a if (anchor and len(sa["long_lines"]) > len(sp["long_lines"])) else neck_p
    split = step4_split_lines(lines, neck["angle_deg"])
    if not split["long_lines"]:
        r["invalid_reason"] = "no_long_lines"; return r
    ei = step5_outer_edges(split["long_lines"], neck["angle_deg"])
    if ei is None:
        r["invalid_reason"] = "no_outer_edges"; return r
    trap = step6_trapezoid(img, ei)
    if trap is None:
        r["invalid_reason"] = "no_trapezoid"; return r
    tok, reasons = validate_trapezoid(trap["corners_px"], img.shape, lms)
    r["trap"] = trap; r["trap_ok"] = tok; r["trap_reasons"] = reasons
    if not tok:
        r["invalid_reason"] = "trap_sanity: " + ", ".join(reasons); return r
    H, H_inv, canon = step6_warp(img, trap["corners_px"])
    r["H"] = H; r["H_inv"] = H_inv; r["canon"] = canon
    nut = step6b_find_nut(canon)
    r["nut"] = nut; r["anchor"] = anchor; r["landmarks"] = lms
    if nut is not None:
        corners_trim = step6c_trim_to_nut(trap["corners_px"], H_inv, nut)
        H2, H2_inv, canon2 = step6_warp(img, corners_trim)
        r["H"] = H2; r["H_inv"] = H2_inv; r["canon"] = canon2
    fret_xs_raw = step7_fret_lines_canonical(r["canon"])
    fret_xs_filt, removed = suppress_finger_pairs(fret_xs_raw)
    r["fret_xs_raw"] = fret_xs_raw
    r["fret_xs_filt"] = fret_xs_filt
    r["removed_pairs"] = removed
    r["ok"] = True
    return r

RESULTS = []
for entry in IMAGES:
    print(f"seed={entry['seed']} | {entry['class']} | {entry['fname']}")
    RESULTS.append(_run_to_frets(entry))
ok_n = sum(1 for r in RESULTS if r["ok"])
print(f"\nKész: {ok_n}/{len(RESULTS)} kép jutott el a fret-detektálásig.")
for r in RESULTS:
    if not r["ok"]:
        print(f"  ✗ seed={r['seed']}: {r.get('invalid_reason','?')}")


## 4. Vizualizáció: Canonical kép + nut

In [ ]:
def _phase_grid(title: str, draw_fn, figsize=(20, 8)):
    """Általános 2×5 grid 10 RESULTS bejegyzéssel.

    draw_fn(ax, result) – egy subplot kirajzolása. Ha a result['ok']=False ÉS a
    fázis amit rajzolunk csak ok eseten értelmezhető, akkor a draw_fn-nek
    kezelnie kell a hiányzó kulcsokat. Az INVALID overlay-t a wrapper rakja rá.
    """
    fig, axes = plt.subplots(2, 5, figsize=figsize)
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for ax, r in zip(axes.ravel(), RESULTS):
        try:
            draw_fn(ax, r)
        except Exception as exc:
            ax.text(0.5, 0.5, f"draw error:\n{exc}", ha="center", va="center",
                    transform=ax.transAxes, color="red", fontsize=8)
        sub_title = f"seed={r['seed']} | {r['class']} | {r['fname'][:14]}"
        ax.set_title(sub_title, fontsize=8)
        ax.axis("off")
        if r.get("invalid_reason"):
            ax.add_patch(plt.Rectangle((0.02, 0.02), 0.96, 0.96, fill=False,
                                        edgecolor="red", lw=2.5,
                                        transform=ax.transAxes))
            ax.text(0.5, 0.94, "INVALID", color="red", fontsize=11,
                    fontweight="bold", ha="center", va="top",
                    transform=ax.transAxes,
                    bbox=dict(facecolor="white", alpha=0.8, pad=2,
                              edgecolor="red"))
    plt.tight_layout()
    plt.show()


def _maybe_imshow(ax, img):
    if img is not None:
        ax.imshow(img)


### 7.7 Canonical warp + nut detektálás

In [ ]:
def _draw_canonical_nut(ax, r):
    canon = r.get("canon")
    if canon is None:
        return
    ax.imshow(bgr2rgb(canon), aspect="auto")
    nut = r.get("nut")
    side_hint = r.get("nut_side_hint")
    if nut is not None:
        col = "magenta" if nut["side"] == "left" else "cyan"
        ax.axvline(nut["nut_x"], color=col, lw=2,
                   label=f"nut@{nut['nut_x']} ({nut['side']})")
    if side_hint:
        ax.text(0.5, 0.95, f"hint={side_hint}", color="white",
                fontsize=7, ha="center", transform=ax.transAxes,
                bbox=dict(facecolor="black", alpha=0.5, pad=1))
    ax.legend(loc="upper right", fontsize=7)

_phase_grid("STEP 6b: Canonical kép + nut", _draw_canonical_nut, figsize=(20, 5))


## 5. Vizualizáció: Fret-jelöltek suppress előtt és után (HIBA 1B)

### 7.8 Fret-jelöltek SUPPRESS előtt és után (HIBA 1B)

In [ ]:
def _draw_fret_candidates(ax, r):
    canon = r.get("canon")
    if canon is None:
        return
    ax.imshow(bgr2rgb(canon), aspect="auto")
    raw  = r.get("fret_xs_raw") or []
    filt = r.get("fret_xs_filt") or []
    removed = r.get("removed_pairs") or []
    filt_set = set(round(x, 2) for x in filt)
    for x in raw:
        is_kept = round(x, 2) in filt_set
        col = "lime" if is_kept else "red"
        ax.axvline(x, color=col, lw=1.2 if is_kept else 0.8,
                   alpha=0.9 if is_kept else 0.6,
                   ls="-" if is_kept else "--")
    ax.text(0.02, 0.92, f"raw={len(raw)}  kept={len(filt)}  pairs_removed={len(removed)}",
            color="white", fontsize=8, transform=ax.transAxes,
            bbox=dict(facecolor="black", alpha=0.6, pad=2))

_phase_grid("STEP 7+7b [HIBA 1B]: Fret jelöltek (zöld=megtartva, piros szaggatott=ujj-él pár)",
            _draw_fret_candidates, figsize=(20, 5))


## 6. Statisztika: nut + fret-jelöltek

In [ ]:
print(f"{'seed':>8} {'class':<8} {'nut':>6} {'raw':>6} {'kept':>6} {'pairs':>6} {'ok':>4}")
print("-" * 50)
for r in RESULTS:
    nut  = r.get("nut")
    raw  = len(r.get("fret_xs_raw") or [])
    kept = len(r.get("fret_xs_filt") or [])
    pairs = len(r.get("removed_pairs") or [])
    nut_str = f"{nut['nut_x']}" if nut else "—"
    ok = "✅" if r["ok"] else "❌"
    print(f"{r['seed']:>8} {r['class']:<8} {nut_str:>6} {raw:>6} {kept:>6} {pairs:>6} {ok:>4}")
